# Manual Review

The AI verification stage deliberately leaves UNCLEAR classifications that should be parsed for worthwhile signals and reviewed. Beyond this there is a need to validate the remaining classifications to capture any poorly made assumptions or bad practices that the AI may have carried out in the verification process.

In [75]:
from pathlib import Path

import pandas as pd

BUSINESS_DATA_FOLDER = Path("../data/business/interim")
VERIFICATION_FOLDER = (BUSINESS_DATA_FOLDER / "ai_verification_v3")
AI_RESULTS_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_results.csv")
V2_MANUAL_REVIEW_PATH = (BUSINESS_DATA_FOLDER / "ai_verification_v2" / "manual_review" / "bakery_manual_review.xlsx")

REVIEW_FOLDER = (VERIFICATION_FOLDER / "manual_review_v3")
REVIEW_FOLDER.mkdir(parents=True, exist_ok=True)

FALSE_POSITIVE_REVIEW_PATH = (REVIEW_FOLDER / "bakery_manual_review_fp_v3.csv")
FALSE_NEGATIVE_REVIEW_PATH = (REVIEW_FOLDER / "bakery_manual_review_fn_v3.csv")
UNCLEAR_REVIEW_PATH = (REVIEW_FOLDER / "bakery_manual_review_unclear_v3.csv")

EXPECTED_TOTAL = 28847

In [76]:
ai_results_raw = pd.read_csv(AI_RESULTS_PATH)

duplicate_count = (ai_results_raw["BusinessNameClean"]
                   .duplicated()
                   .sum())

if duplicate_count:
    print(f"Duplicate result rows found: {duplicate_count}")

ai_results = (ai_results_raw
              .sort_values("VerificationDateTime")
              .drop_duplicates(subset="BusinessNameClean",
                               keep="last")
              .sort_values("BakeryRank")
              .reset_index(drop=True))

print(f"Loaded from: {AI_RESULTS_PATH}")
print(f"Rows completed out of total: {len(ai_results)}/{EXPECTED_TOTAL}")

if len(ai_results) != EXPECTED_TOTAL:
    raise RuntimeError(f"AI verification is not complete. Notebook 09 stops here.")

print("\nAI verification is complete. Manual-review diagnostics can proceed.")

Duplicate result rows found: 10
Loaded from: ..\data\business\interim\ai_verification_v3\bakery_ai_verification_results.csv
Rows completed out of total: 28847/28847

AI verification is complete. Manual-review diagnostics can proceed.


Checking duplicated cases

In [77]:
duplicate_rows = ai_results_raw[ai_results_raw["BusinessNameClean"].duplicated(keep=False)]

duplicate_conflicts = (duplicate_rows
                       .groupby("BusinessNameClean")["AIClass"]
                       .nunique())

duplicate_conflicts = duplicate_conflicts[duplicate_conflicts > 1]

print(f"Duplicate businesses with conflicting classifications: {len(duplicate_conflicts)}")

Duplicate businesses with conflicting classifications: 1


# Basic validation of dataset

In [4]:
required_columns = ["BusinessNameClean",
                    "BusinessName",
                    "FHRSIDRep",
                    "BusinessType",
                    "PostCode",
                    "LocalAuthorityName",
                    "Address",
                    "BakeryRank",
                    "BakeryScore",
                    "StoreCount",
                    "LocationMatch",
                    "AIClass",
                    "AIReason"]

missing_columns = [column
                   for column in required_columns
                   if column not in ai_results.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if ai_results["BakeryRank"].duplicated().any():
    raise ValueError("Duplicate BakeryRank values found.")

if ai_results["BusinessNameClean"].duplicated().any():
    raise ValueError("Duplicate BusinessNameClean values found.")

# Checking that the AI verified according to prompt rules
valid_classes = {"CORE_BAKERY",
                 "BAKERY_CAFE",
                 "GROCER_BAKERY",
                 "NON_BAKERY",
                 "UNCLEAR"}

invalid_classes = (set(ai_results["AIClass"].dropna()) - valid_classes)
if invalid_classes:
    raise ValueError(f"Invalid AIClass values: {invalid_classes}")

valid_locations = {"YES", "UNCLEAR"}

invalid_locations = (set(ai_results["LocationMatch"].dropna()) - valid_locations)
if invalid_locations:
    raise ValueError(f"Invalid LocationMatch values: {invalid_locations}")

location_conflicts = (ai_results["LocationMatch"].eq("UNCLEAR")
                      & ~ai_results["AIClass"].eq("UNCLEAR"))

if location_conflicts.any():
    raise ValueError("LocationMatch=UNCLEAR rows found with a non-UNCLEAR AIClass.")

missing_reasons = (ai_results["AIReason"]
                   .fillna("")
                   .str.strip()
                   .eq(""))

if missing_reasons.any():
    raise ValueError(f"{missing_reasons.sum()} AIReason values are blank.")

print("Verification dataset checks passed.")

Verification dataset checks passed.


In [5]:
display(ai_results["AIClass"]
        .value_counts()
        .rename("Count"))

display(pd.crosstab(ai_results["LocationMatch"],
                    ai_results["AIClass"],
                    margins=True))

AIClass
NON_BAKERY       19883
UNCLEAR           5675
GROCER_BAKERY     1334
BAKERY_CAFE       1320
CORE_BAKERY        635
Name: Count, dtype: int64

AIClass,BAKERY_CAFE,CORE_BAKERY,GROCER_BAKERY,NON_BAKERY,UNCLEAR,All
LocationMatch,,,,,,
UNCLEAR,0,0,0,0,5539,5539
YES,1320,635,1334,19883,136,23308
All,1320,635,1334,19883,5675,28847


# Targeted review

The manual review aims to be selective instead of being done exhaustitively. Looking at the AI classifications and AI reasoning allowed for some recurring patterns to be observed that produces errors. These review was therefore refined to focus on where manual checks would lead to changes in whether the business is included in the final bakery dataset or removed.

Potential false positives were first checked for among the positive bakery classes. In GROCER_BAKERY classifications particular attention was given where reasoning seemed weak or speculative, whereas in BAKERY_CAFE classifications some reasoning indicated a lack of bakery range, finally with CORE_BAKERY classifications there was reasoning that explicitly broke the single-product exclusion rule that needs to be addressed. These will all be manually checked using keywords to extract relevant establishments using the AIReason column.

For potential false negatives from UNCLEAR and NON_BAKERY classifications, only business with strong bakery signals, bakery-related names and bakery evidence in the AI reasoning were checked as these might support one of the positive bakery classes.

The diagnostic rules are used to construct targeted review queues, while some decisions and previously completed V2 manual decisions are also carried forward to final review files.

In [6]:
diagnostics = ai_results.copy()

diagnostics["FormattedReason"] = (diagnostics["AIReason"]
                                  .fillna("")
                                  .str.lower())

diagnostics["FormattedName"] = (diagnostics["BusinessNameClean"]
                                 .fillna("")
                                 .str.lower())

reason = diagnostics["FormattedReason"]
name = diagnostics["FormattedName"]

def contains(series, pattern):
    return series.str.contains(pattern, regex=True, na=False)

def read_review_csv(path):
    try:
        return pd.read_csv(path, keep_default_na=False, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, keep_default_na=False, encoding="cp1252")

In [7]:
MANUAL_COLUMNS = ["ManualClass", "ManualReason", "ManualSourceURL"]

def preserve_review_work(review_df, path):

    if not path.exists():
        return review_df

    existing = read_review_csv(path)

    existing_manual = (existing[["BusinessNameClean"] + MANUAL_COLUMNS]
                       .drop_duplicates(subset="BusinessNameClean", keep="last"))

    review_df = (review_df
                 .drop(columns=MANUAL_COLUMNS)
                 .merge(existing_manual, on="BusinessNameClean", how="left", validate="one_to_one"))

    review_df[MANUAL_COLUMNS] = (review_df[MANUAL_COLUMNS].fillna(""))

    return review_df

In [8]:
v2_manual = pd.read_excel(V2_MANUAL_REVIEW_PATH)

v2_manual = (v2_manual[["BusinessNameClean", "ManualVerdict", "ManualNote"]]
             .dropna(subset=["ManualVerdict"])
             .drop_duplicates(subset="BusinessNameClean", keep="last")
             .rename(columns={"ManualVerdict": "V2ManualVerdict",
                              "ManualNote": "V2ManualNote"}))

diagnostics = diagnostics.merge(v2_manual,
                                on="BusinessNameClean",
                                how="left",
                                validate="one_to_one")

print(diagnostics["V2ManualVerdict"].value_counts(dropna=False))

V2ManualVerdict
NaN           28239
NOT_BAKERY      391
BAKERY          215
MIXED             2
Name: count, dtype: int64


In [9]:
positive_class = diagnostics["AIClass"].isin(["CORE_BAKERY",
                                              "BAKERY_CAFE",
                                              "GROCER_BAKERY"])

diagnostics["FP_V2_MANUAL_NOT_BAKERY"] = (positive_class
                                          & diagnostics["V2ManualVerdict"].eq("NOT_BAKERY"))

# Creating false-positive review dataset



In [10]:
#Core_Bakery Patterns

SPECIALISED_PRODUCT_PATTERN = (r"(?:celebration cakes?|wedding cakes?|birthday cakes?|bespoke cakes?|"
                               r"custom(?:ised|ized)? cakes?|cookies?\b|doughnuts?\b|donuts?\b|"
                               r"pretzels?\b|cinnamon rolls?\b|cinnamon buns?\b|macarons?\b|waffles?\b|"
                               r"crepes?\b|brownies?\b|cupcakes?\b)")

CORE_BAKERY_PATTERN = (r"\b(?:bread|breads|sourdough|baguette|baguettes|loaf|loaves|naan|"
                       r"roti|pastry|pastries|croissant|croissants|patisserie|pâtisserie)\b")

In [11]:
#Cafe_Bakery Patterns

CAFE_BAKERY_PATTERN = (r"\b(?:bakery|bakeries|bakes?|freshly baked|baked goods|bread|breads|"
                       r"sourdough|pastry|pastries|croissant|croissants|muffin|muffins|"
                       r"cake|cakes|bun|buns|roll|rolls|bagel|bagels|baguette|baguettes|"
                       r"focaccia|cookie|cookies|patisserie|pâtisserie|scone|scones)\b")

In [12]:
#Grocer_Bakery Patterns

STRONG_GROCER_BAKERY_PATTERN = (r"(?:in[- ]store bakery|dedicated bakery|bakery section|"
                                r"bakery counter|bakery concession|greggs concession|"
                                r"baked in[- ]store)")

LACKING_EVIDENCE_PATTERN = (r"(?:no (?:(?:specific|affirmative|clear|direct|explicit) )?evidence|no specific information|"
                            r"(?:specific )?bakery (?:provision|evidence).{0,60}(?:isn['’]?t|is not) detailed|"
                            r"not explicitly stated|without (?:specific|affirmative|clear|direct|explicit) evidence)")

SPECULATIVE_GROCER_PATTERN = (r"\b(?:likely|probably|plausible|typically|generally|assumed|presumably|could|may|might)\b")

In [13]:
#Establishments that should have been excluded/lack of location evidence

FALSE_POSITIVE_EXCLUSION_PATTERN = (r"(?:home[- ]based|private address|home kitchen|dark kitchen|"
                                    r"\bonline business\b|\bonline .*store selling\b|"
                                    r"^online shop\b|wholesale[- ]only|production[- ]only|"
                                    r"permanently closed|ceased trading|main bakery.*closed|"
                                    r"bakery operation.*no longer active)")

INCONCLUSIVE_LOCATION_PATTERN = (r"(?:similar business|precedent of similar|"
                                 r"nearby .*(?:bakery|patisserie)|another .*menu)")

AUTO_NON_BAKERY_PATTERN = (r"(?:private address|private residence|"
                           r"home[- ]based|home bakery|home baker|home kitchen|"
                           r"residential address|residential property|residential flat)")

HOME_BAKERY_NAME_PATTERN = (r"\b(?:home bakery|home baker|home[- ]based)\b")

V2_SAFE_NON_BAKERY_PATTERN = (r"(?:private address|private residence|residential address|"
                              r"residential property|residential flat|"
                              r"home[- ]based|home bakery|home baker|home kitchen|"
                              r"online[- ]only|delivery[- ]only|"
                              r"wholesale[- ]only|production[- ]only|"
                              r"mobile caterer|no fixed address|"
                              r"permanently closed|ceased trading|"
                              r"cake[- ]only|cookie[- ]only|doughnut[- ]only|donut[- ]only)")

# Creating False-Positive Flags

In [14]:
core_class = diagnostics["AIClass"].eq("CORE_BAKERY")
cafe_class = diagnostics["AIClass"].eq("BAKERY_CAFE")
grocer_class = diagnostics["AIClass"].eq("GROCER_BAKERY")

positive_class = diagnostics["AIClass"].isin(["CORE_BAKERY", "BAKERY_CAFE", "GROCER_BAKERY"])

lacking_evidence = contains(reason, LACKING_EVIDENCE_PATTERN)

diagnostics["FP_GROCER_WEAK_EVIDENCE"] = (grocer_class
                                          & (lacking_evidence 
                                              | (contains(reason, SPECULATIVE_GROCER_PATTERN) 
                                                 & ~contains(reason, STRONG_GROCER_BAKERY_PATTERN))))

diagnostics["FP_CAFE_WEAK_EVIDENCE"] = (cafe_class
                                        & (lacking_evidence 
                                           | ~contains(reason, CAFE_BAKERY_PATTERN)))

diagnostics["FP_CORE_SPECIALISED_PRODUCT"] = (core_class
                                              & contains(reason, SPECIALISED_PRODUCT_PATTERN)
                                              & ~contains(reason, CORE_BAKERY_PATTERN))


diagnostics["FP_EXCLUSION"] = (positive_class
                               & (contains(reason, FALSE_POSITIVE_EXCLUSION_PATTERN)
                                  | contains(reason, AUTO_NON_BAKERY_PATTERN)
                                  | contains(name, HOME_BAKERY_NAME_PATTERN)))

diagnostics["FP_INCONCLUSIVE_LOCATION"] = (positive_class
                                           & contains(reason, INCONCLUSIVE_LOCATION_PATTERN))

In [15]:
false_positive_flags = ["FP_V2_MANUAL_NOT_BAKERY",
                        "FP_GROCER_WEAK_EVIDENCE",
                        "FP_CAFE_WEAK_EVIDENCE",
                        "FP_CORE_SPECIALISED_PRODUCT",
                        "FP_EXCLUSION",
                        "FP_INCONCLUSIVE_LOCATION"]

display(diagnostics[false_positive_flags]
        .sum()
        .sort_values(ascending=False)
        .rename("Count"))

false_positive_reviews = diagnostics[false_positive_flags].any(axis=1)

print(f"Unique false-positive review cases: {false_positive_reviews.sum()}")

FP_GROCER_WEAK_EVIDENCE        305
FP_V2_MANUAL_NOT_BAKERY         82
FP_CAFE_WEAK_EVIDENCE           38
FP_CORE_SPECIALISED_PRODUCT     25
FP_EXCLUSION                    11
FP_INCONCLUSIVE_LOCATION         6
Name: Count, dtype: int64

Unique false-positive review cases: 453


In [16]:
false_positive_review = diagnostics[false_positive_reviews].copy()

false_positive_review["ReviewFlags"] = (false_positive_review[false_positive_flags]
                                        .apply(lambda row: "; ".join([flag for flag in false_positive_flags 
                                                                      if row[flag]]), axis=1))

In [84]:
review_columns = ["BakeryRank", "BusinessName", "BusinessNameClean",
                  "BusinessType", "Address", "PostCode", "LocalAuthorityName",
                  "FHRSIDRep", "StoreCount", "BakeryScore", "LocationMatch", "AIClass", 
                  "V2ManualVerdict", "AIReason", "V2ManualNote", "ReviewFlags"]

false_positive_review = (false_positive_review[review_columns]
                         .sort_values("BakeryRank")
                         .reset_index(drop=True))

false_positive_review["ManualClass"] = ""
false_positive_review["ManualReason"] = ""
false_positive_review["ManualSourceURL"] = ""

false_positive_review = preserve_review_work(false_positive_review, FALSE_POSITIVE_REVIEW_PATH)

if FALSE_POSITIVE_REVIEW_PATH.exists():
    existing_fp_review = read_review_csv(FALSE_POSITIVE_REVIEW_PATH)

    completed_fp = existing_fp_review[existing_fp_review["ManualClass"]
                                      .str.strip()
                                      .ne("")].copy()

    completed_fp = completed_fp[["BusinessNameClean", "ManualClass", "ManualReason", "ManualSourceURL"]]

    false_positive_review = (false_positive_review
                             .drop(columns=["ManualClass", "ManualReason", "ManualSourceURL"])
                             .merge(completed_fp, on="BusinessNameClean", how="left"))

    false_positive_review[["ManualClass", "ManualReason", "ManualSourceURL"]] = (
        false_positive_review[["ManualClass", "ManualReason", "ManualSourceURL"]]
        .fillna(""))

print(f"Updated {len(false_positive_review)} false-positive review rows.")

print(f"Completed: {false_positive_review['ManualClass'].str.strip().ne("").sum()}")

Updated 453 false-positive review rows.
Completed: 453


In [85]:
still_unreviewed = (false_positive_review["ManualClass"]
                    .str.strip()
                    .eq(""))

v2_note = (false_positive_review["V2ManualNote"]
           .fillna("")
           .str.lower())

v2_safe_negative = (still_unreviewed
                    & false_positive_review["V2ManualVerdict"].eq("NOT_BAKERY")
                    & contains(v2_note, V2_SAFE_NON_BAKERY_PATTERN))

grocer_auto = (false_positive_review["ReviewFlags"]
                .str.contains("FP_GROCER_WEAK_EVIDENCE", na=False) 
                & still_unreviewed)
    
false_positive_review.loc[grocer_auto, "ManualClass"] = "NON_BAKERY"
false_positive_review.loc[grocer_auto, "ManualReason"
                          ] = "No location-specific evidence of bakery provision."

exclusion_auto = (still_unreviewed 
                  & false_positive_review["ReviewFlags"]
                  .str.contains("FP_EXCLUSION", na=False))
    
false_positive_review.loc[exclusion_auto, "ManualClass"] = "NON_BAKERY"
false_positive_review.loc[exclusion_auto, "ManualReason"
                          ] = "Current evidence meets an explicit exclusion criterion."

false_positive_review.loc[v2_safe_negative,"ManualClass"] = "NON_BAKERY"

false_positive_review.to_csv(FALSE_POSITIVE_REVIEW_PATH, index=False, encoding="utf-8-sig")

# Creating False-Negative review dataset

In [64]:
STRONG_BAKERY_NAME_PATTERN = (r"\b(?:bakery|bakeries|bakers|bakehouse|boulangerie|"
                              r"patisserie|pâtisserie|bread|sourdough|bagel|bagels|"
                              r"loaf|flour|pastry|pastries)\b")

NEGATIVE_LOCATION_PATTERN = (r"(?:home[- ]based|home baker|home bakery|home kitchen|cottage bakery|"
                             r"private address|private residence|residential address|"
                             r"residential flat|flat address|"
                             r"online[- ]only|delivery[- ]only|online cake shop|online retailer|"
                             r"online business|online shop|mobile caterer|"
                             r"wholesale bakery|wholesale supplier|wholesale[- ]only|"
                             r"production[- ]only|production kitchen|"
                             r"primarily an online|primarily a home|"
                             r"no fixed address|collection only)")

GROCER_RECOVERY_PATTERN = (r"bakery.*(?:department|section|category)")

BAKERY_CHAIN_PATTERN = (
    r"\b(?:starbucks|costa|caff[eè] nero|"
    r"pret(?: a manger)?|upper crust|"
    r"greggs|gail['’]?s)\b"
)

In [65]:
unclear_class = diagnostics["AIClass"].eq("UNCLEAR")
non_bakery_class = diagnostics["AIClass"].eq("NON_BAKERY")

strong_bakery_name = contains(name, STRONG_BAKERY_NAME_PATTERN)
clear_negative = contains(reason, NEGATIVE_LOCATION_PATTERN)
single_product = contains(reason, SPECIALISED_PRODUCT_PATTERN)

address_available = (diagnostics["Address"].fillna("").str.strip().ne(""))

full_postcode = (diagnostics["PostCode"].fillna("")
                 .str.upper()
                 .str.strip()
                 .str.match(r"^[A-Z]{1,2}\d[A-Z\d]?\s*\d[A-Z]{2}$"))

usable_location = (address_available | full_postcode)

out_class = diagnostics["AIClass"].isin(["NON_BAKERY", "UNCLEAR"])

In [66]:
diagnostics["FN_UNCLEAR_BAKERY"] = (unclear_class
                                    & strong_bakery_name
                                    & usable_location
                                    & ~clear_negative
                                    & ~single_product
                                    & ~diagnostics["BusinessType"].eq("Mobile caterer"))

diagnostics["FN_NON_BAKERY_STRONG_NAME"] = (non_bakery_class
                                            & strong_bakery_name
                                            & usable_location
                                            & ~clear_negative
                                            & ~single_product
                                            & ~diagnostics["BusinessType"].eq("Mobile caterer"))

diagnostics["FN_UNCLEAR_GROCER"] = (unclear_class
                                    & diagnostics["LocationMatch"].eq("YES")
                                    & contains(reason, GROCER_RECOVERY_PATTERN)
                                    & ~clear_negative)

#Very specific observation found in non-bakeries falsely classified
diagnostics["FN_NON_BAKERY_NOT_BAKERY_LED"] = (non_bakery_class
                                               & strong_bakery_name
                                               & contains(reason, r"\bbut not (?:a )?bakery[- ]led\b"))

diagnostics["FN_CHAIN_CONSISTENCY"] = (out_class
                                       & diagnostics["LocationMatch"].eq("YES")
                                       & usable_location
                                       & contains(name, BAKERY_CHAIN_PATTERN))

In [67]:
false_negative_flags = ["FN_UNCLEAR_BAKERY", 
                        "FN_UNCLEAR_GROCER", 
                        "FN_NON_BAKERY_NOT_BAKERY_LED", 
                        "FN_NON_BAKERY_STRONG_NAME",
                        "FN_CHAIN_CONSISTENCY"]

display(diagnostics[false_negative_flags]
        .sum().rename("Count"))

false_negative_reviews = diagnostics[false_negative_flags].any(axis=1)

print(f"Unique false-negative review cases: {false_negative_reviews.sum()}")

FN_UNCLEAR_BAKERY               24
FN_UNCLEAR_GROCER                1
FN_NON_BAKERY_NOT_BAKERY_LED     2
FN_NON_BAKERY_STRONG_NAME       95
FN_CHAIN_CONSISTENCY            11
Name: Count, dtype: int64

Unique false-negative review cases: 131


In [86]:
false_negative_review = diagnostics[false_negative_reviews].copy()

false_negative_review["ReviewFlags"] = (false_negative_review[false_negative_flags]
                                        .apply(lambda row: "; ".join(
                                            [flag for flag in false_negative_flags if row[flag]]), axis=1))

false_negative_review = (false_negative_review[review_columns]
                         .sort_values("BakeryRank")
                         .reset_index(drop=True))

false_negative_review["ManualClass"] = ""
false_negative_review["ManualReason"] = ""
false_negative_review["ManualSourceURL"] = ""

false_negative_review = preserve_review_work(false_negative_review, FALSE_NEGATIVE_REVIEW_PATH)

manual_columns = ["ManualClass", "ManualReason", "ManualSourceURL"]

# Preserve manual work already completed
if FALSE_NEGATIVE_REVIEW_PATH.exists():

    existing_fn_review = pd.read_csv(FALSE_NEGATIVE_REVIEW_PATH, keep_default_na=False)

    completed_fn = (existing_fn_review[existing_fn_review["ManualClass"]
                                       .str.strip()
                                       .ne("")]
                                       .drop_duplicates(subset="BusinessNameClean", keep="last"))

    manual_values = completed_fn[["BusinessNameClean"] + manual_columns]

    # Restoring existing manual decisions for businesses in current review queue
    false_negative_review = (false_negative_review
                             .drop(columns=manual_columns)
                             .merge(manual_values, on="BusinessNameClean", how="left"))

    false_negative_review[manual_columns] = (false_negative_review[manual_columns].fillna(""))

    # Also retain completed manual reviews that no longer match the latest diagnostic rules
    completed_old_only = completed_fn[~completed_fn["BusinessNameClean"]
                                      .isin(false_negative_review["BusinessNameClean"])].copy()

    if len(completed_old_only):
        completed_old_only = completed_old_only.reindex(columns=false_negative_review.columns, fill_value="")

        false_negative_review = pd.concat([false_negative_review, completed_old_only], ignore_index=True)

# Reuse safe V2 manual NON_BAKERY decisions
still_unreviewed = (false_negative_review["ManualClass"]
                    .str.strip()
                    .eq(""))

v2_note = (false_negative_review["V2ManualNote"]
           .fillna("")
           .str.lower())

v2_safe_negative = (still_unreviewed
                    & false_negative_review["V2ManualVerdict"].eq("NOT_BAKERY")
                    & contains(v2_note, V2_SAFE_NON_BAKERY_PATTERN))

false_negative_review.loc[v2_safe_negative, "ManualClass"] = "NON_BAKERY"

false_negative_review = (false_negative_review
                         .sort_values("BakeryRank")
                         .reset_index(drop=True))

false_negative_review.to_csv(FALSE_NEGATIVE_REVIEW_PATH, index=False, encoding="utf-8-sig")

print(f"Updated {len(false_negative_review)} false-negative review rows.")

print(f"Completed: {false_negative_review['ManualClass'].str.strip().ne('').sum()}")

Updated 131 false-negative review rows.
Completed: 131


# Additional Unclear Reviews

In [78]:
unclear_class = diagnostics["AIClass"].eq("UNCLEAR")

immediate_exclusion = (contains(reason, NEGATIVE_LOCATION_PATTERN)
                       | contains(reason, AUTO_NON_BAKERY_PATTERN)
                       | diagnostics["BusinessType"].eq("Mobile caterer"))

additional_unclear_reviews = (unclear_class
                              & diagnostics["LocationMatch"].eq("YES")
                              & ~immediate_exclusion
                              & ~false_negative_reviews)

print(f"Additional location-confirmed UNCLEAR cases: {additional_unclear_reviews.sum()}")

Additional location-confirmed UNCLEAR cases: 112


In [87]:
unclear_review = diagnostics[additional_unclear_reviews].copy()

unclear_review["ReviewFlags"] = "UNCLEAR_LOCATION_CONFIRMED"

unclear_review = (unclear_review[review_columns]
                  .sort_values("BakeryRank")
                  .reset_index(drop=True))

unclear_review["ManualClass"] = ""
unclear_review["ManualReason"] = ""
unclear_review["ManualSourceURL"] = ""

unclear_review = preserve_review_work(unclear_review, UNCLEAR_REVIEW_PATH)

manual_columns = ["ManualClass", "ManualReason", "ManualSourceURL"]

# Preserve any UNCLEAR reviews already completed
if UNCLEAR_REVIEW_PATH.exists():
    existing_unclear_review = pd.read_csv(UNCLEAR_REVIEW_PATH, keep_default_na=False)

    completed_unclear = (existing_unclear_review[existing_unclear_review["ManualClass"]
                                                 .str.strip()
                                                 .ne("")]
                                                 .drop_duplicates(subset="BusinessNameClean", keep="last"))

    manual_values = completed_unclear[["BusinessNameClean"] + manual_columns]

    unclear_review = (unclear_review
                      .drop(columns=manual_columns)
                      .merge(manual_values,
                             on="BusinessNameClean",
                             how="left"))

    unclear_review[manual_columns] = (unclear_review[manual_columns].fillna(""))

    # Keep completed work even if the selection changes later
    completed_old_only = completed_unclear[~completed_unclear["BusinessNameClean"]
                                           .isin(unclear_review["BusinessNameClean"])].copy()

    if len(completed_old_only):
        completed_old_only = completed_old_only.reindex(columns=unclear_review.columns, fill_value="")

        unclear_review = pd.concat([unclear_review, completed_old_only], ignore_index=True)

# Reuse safe V2 manual NON_BAKERY decisions
still_unreviewed = (unclear_review["ManualClass"]
                    .str.strip()
                    .eq(""))

v2_note = (unclear_review["V2ManualNote"]
           .fillna("")
           .str.lower())

v2_safe_negative = (still_unreviewed
                    & unclear_review["V2ManualVerdict"].eq("NOT_BAKERY")
                    & contains(v2_note, V2_SAFE_NON_BAKERY_PATTERN))

unclear_review.loc[v2_safe_negative, "ManualClass"] = "NON_BAKERY"

unclear_review = (unclear_review
                  .sort_values("BakeryRank")
                  .reset_index(drop=True))

unclear_review.to_csv(UNCLEAR_REVIEW_PATH, index=False, encoding="utf-8-sig")

print(f"Updated {len(unclear_review)} additional UNCLEAR review rows.")
print(f"Completed: {unclear_review['ManualClass'].str.strip().ne('').sum()}")

Updated 112 additional UNCLEAR review rows.
Completed: 112


# Overview

In [88]:
fp_review = read_review_csv(FALSE_POSITIVE_REVIEW_PATH)
fn_review = read_review_csv(FALSE_NEGATIVE_REVIEW_PATH)
unclear_review = read_review_csv(UNCLEAR_REVIEW_PATH)

fp_complete = fp_review["ManualClass"].str.strip().ne("")
fn_complete = fn_review["ManualClass"].str.strip().ne("")
unclear_complete = (unclear_review["ManualClass"].str.strip().ne(""))

print(f"False positives: {fp_complete.sum()}/{len(fp_review)} reviewed")
print(f"False negatives: {fn_complete.sum()}/{len(fn_review)} reviewed")
print(f"Additional UNCLEARs: {unclear_complete.sum()}/{len(unclear_review)} reviewed")

total_done = fp_complete.sum() + fn_complete.sum() + unclear_complete.sum()
total_reviews = len(fp_review) + len(fn_review) + len(unclear_review)

print(f"\nOverall: {total_done}/{total_reviews} reviewed")

False positives: 453/453 reviewed
False negatives: 131/131 reviewed
Additional UNCLEARs: 112/112 reviewed

Overall: 696/696 reviewed


# Saving Manual Reviews

In [90]:
FINAL_REVIEWED_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_reviewed.csv")

review_sets = pd.concat([fp_review.assign(ReviewSet="FALSE_POSITIVE"),
                         fn_review.assign(ReviewSet="FALSE_NEGATIVE"),
                         unclear_review.assign(ReviewSet="ADDITIONAL_UNCLEAR")],
                         ignore_index=True)

# Every selected review should first be completed
missing_manual = (review_sets["ManualClass"]
                  .str.strip()
                  .eq(""))

if missing_manual.any():
    raise ValueError(f"{missing_manual.sum()} manual reviews are incomplete.")

invalid_manual = (set(review_sets["ManualClass"]) - valid_classes)

if invalid_manual:
    raise ValueError(f"Invalid ManualClass values: {invalid_manual}")

# A business should not have multiple manual decisions
duplicate_reviews = (review_sets["BusinessNameClean"]
                     .duplicated(keep=False))

if duplicate_reviews.any():
    raise ValueError("Businesses appear in more than one review dataset.")

# Apply manual decisions over the original AI classifications
final_results = ai_results.merge(
    review_sets[["BusinessNameClean",
                 "ManualClass",
                 "ManualReason",
                 "ManualSourceURL",
                 "ReviewFlags",
                 "ReviewSet"]],
                 on="BusinessNameClean",
                 how="left",
                 validate="one_to_one")

final_results["FinalClass"] = (
    final_results["ManualClass"]
    .where(final_results["ManualClass"]
           .fillna("")
           .str.strip()
           .ne(""),
           final_results["AIClass"]))

if len(final_results) != EXPECTED_TOTAL:
    raise ValueError("Final dataset row count does not match expected total.")

if not set(final_results["FinalClass"]).issubset(valid_classes):
    raise ValueError("Invalid FinalClass values found.")

final_results.to_csv(FINAL_REVIEWED_PATH, index=False, encoding="utf-8-sig")

print(f"Saved final reviewed dataset to: {FINAL_REVIEWED_PATH}")

display(final_results["FinalClass"]
        .value_counts()
        .rename("Count"))

Saved final reviewed dataset to: ..\data\business\interim\ai_verification_v3\bakery_ai_verification_reviewed.csv


FinalClass
NON_BAKERY       20353
UNCLEAR           5590
BAKERY_CAFE       1298
GROCER_BAKERY      996
CORE_BAKERY        610
Name: Count, dtype: int64